# Mission 8：建立小考成績欄位
#### Knowledge Point
在 DataFrame 新增欄位：df["新欄位名"] = 計算結果

向量化運算：欄位之間可直接進行數學運算

分行撰寫公式以利閱讀
選取多欄位：df[["欄A", "欄B"]]
#### Markdown Cell
我們根據 Mission 4 的公式，為 60 位學生一次計算成績，並加入「隨機誤差」模擬真實世界的不可預測因素。
#### 公式：
quiz_score = 50 + 3*study_hours + 2*sleep_hours - 1.2*screen_time_hours + 0.15*attendance_rate + 隨機誤差(平均0, 標準差5)

最後限制在 0~100 分，四捨五入成整數。

In [10]:
# W6-HW-1
# 匯入 numpy 與 pandas，並給予常用別名
import numpy as np
import pandas as pd

# 檢查套件是否成功載入（顯示版本號）
print("numpy 版本：", np.__version__)
print("pandas 版本：", pd.__version__)

# 建立一個可重現的亂數產生器
random_seed = 42
rng = np.random.default_rng(random_seed)
print("亂數產生器已建立，種子為：", random_seed)
# ------------------------------
# 1️⃣ 設定參數與亂數種子
# ------------------------------
student_count = 60                     # 學生人數
random_seed = 42
rng = np.random.default_rng(random_seed)

# ------------------------------
# 2️⃣ 建立各欄位的模擬值
# ------------------------------
# 學生編號：從 1 到 60
student_id = np.arange(1, student_count + 1)

# 系別：從三個選項中隨機選擇
departments_list = ["資訊管理", "企業管理", "金融管理"]
department = rng.choice(departments_list, size=student_count)

# 睡眠時數：常態分布 → 四捨五入到小數點後一位
sleep_hours_raw = rng.normal(loc=6.7, scale=1.1, size=student_count)
sleep_hours = np.round(sleep_hours_raw, 1)

# 螢幕使用時數：常態分布 → clip 到 1~10 小時 → 四捨五入
screen_time_raw = rng.normal(loc=5.0, scale=1.5, size=student_count)
screen_time = np.clip(screen_time_raw, 1.0, 10.0)
screen_time = np.round(screen_time, 1)

# 讀書時數：常態分布 → clip 到 0.5~6 小時 → 四捨五入
study_hours_raw = rng.normal(loc=2.5, scale=1.0, size=student_count)
study_hours = np.clip(study_hours_raw, 0.5, 6.0)
study_hours = np.round(study_hours, 1)

# 出席率：常態分布 → clip 到 60~100 → 四捨五入成整數
attendance_raw = rng.normal(loc=88.0, scale=8.0, size=student_count)
attendance = np.clip(attendance_raw, 60, 100)
attendance = np.round(attendance).astype(int)   # 轉成整數

# ------------------------------
# 3️⃣ 組合成 pandas DataFrame
# ------------------------------
df = pd.DataFrame({
    "student_id": student_id,
    "department": department,
    "sleep_hours": sleep_hours,
    "screen_time_hours": screen_time,
    "study_hours": study_hours,
    "attendance_rate": attendance
})

# ------------------------------
# 4️⃣ 檢查前五筆資料
# ------------------------------
print("前五筆模擬資料：")
display(df.head(60))   # 在 Jupyter/Colab 中使用 display 會得到更好的表格顯示
# ------------------------------
# 1️⃣ 計算各項影響（保持中間變數，方便觀察）
# ------------------------------
base = 50
study_effect = 3 * df["study_hours"]
sleep_effect = 2 * df["sleep_hours"]
screen_effect = -1.2 * df["screen_time_hours"]
attendance_effect = 0.15 * df["attendance_rate"]

# ------------------------------
# 2️⃣ 加入隨機誤差（平均 0，標準差 5）
# ------------------------------
random_error = rng.normal(loc=0.0, scale=5.0, size=student_count)

# ------------------------------
# 3️⃣ 總分（未限制）
# ------------------------------
raw_quiz = base + study_effect + sleep_effect + screen_effect + attendance_effect + random_error

# ------------------------------
# 4️⃣ 限制在 0~100 並四捨五入成整數
# ------------------------------
quiz_score = np.clip(raw_quiz, 0, 100)   # 先裁剪
quiz_score = np.round(quiz_score).astype(int)   # 四捨五入並轉成整數

# ------------------------------
# 5️⃣ 寫回 DataFrame
# ------------------------------
df["quiz_score"] = quiz_score

# ------------------------------
# 6️⃣ 檢查前五筆資料（只顯示關心的欄位）
# ------------------------------
display(df[
    ["student_id",
     "study_hours",
     "sleep_hours",
     "screen_time_hours",
     "attendance_rate",
     "quiz_score"]
].head())

numpy 版本： 2.1.3
pandas 版本： 2.2.3
亂數產生器已建立，種子為： 42
前五筆模擬資料：


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate
0,1,資訊管理,9.1,6.0,2.3,95
1,2,金融管理,6.3,4.9,1.5,77
2,3,企業管理,6.1,4.4,0.8,88
3,4,企業管理,5.8,4.9,2.0,84
4,5,企業管理,7.4,2.5,2.4,85
5,6,金融管理,7.9,2.8,4.3,96
6,7,資訊管理,6.6,3.0,2.6,92
7,8,金融管理,5.8,3.5,3.5,99
8,9,資訊管理,5.8,5.6,2.0,87
9,10,資訊管理,7.4,3.6,1.3,82


,student_id,study_hours,sleep_hours,screen_time_hours,attendance_rate,quiz_score
0,1,2.3,9.1,6.0,95,79
1,2,1.5,6.3,4.9,77,74
2,3,0.8,6.1,4.4,88,72
3,4,2.0,5.8,4.9,84,85
4,5,2.4,7.4,2.5,85,90
